In [6]:
from pathlib import Path
import numpy as np
import torch
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim

from mads_datasets import datatools
import mltrainer
mltrainer.__version__

'0.2.7'

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 


In [9]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-03-06 14:36:37.897 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\Alex\.cache\mads_datasets\gestures
100%|██████████| 651/651 [00:00<00:00, 1614.40it/s]


In [11]:
len(train), len(valid)

(81, 20)

In [12]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 30, 3]),
 tensor([14, 14, 17,  9, 10, 12,  2, 11,  4,  2, 16, 14, 16,  6, 10,  8,  4,  7,
         10, 13, 12,  4, 12, 11, 18, 11,  8, 19,  6, 10, 11,  7]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?


# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

In [13]:
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [14]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [15]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [16]:
accuracy(y, yhat)

0.125

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [17]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [18]:
yhat[0]

tensor([-0.1797, -0.1265,  0.2029,  0.0115, -0.1380,  0.1016, -0.1168, -0.1210,
        -0.0624, -0.2062,  0.2213, -0.0633,  0.1287, -0.1766,  0.0912,  0.0393,
         0.1670,  0.0125, -0.0032, -0.0289], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 

In [19]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(2.9806, grad_fn=<NllLossBackward0>)

In [20]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

using cpu


Set up the settings for the trainer and the different types of logging you want

In [21]:
settings = TrainerSettings(
    epochs=3, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings

epochs: 3
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

In [22]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [23]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)


In [24]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "modelname-here")
    mlflow.set_tag("dev", "your-name-here")
    config = ModelConfig(
        input_size=3,
        hidden_size=64,
        num_layers=1,
        output_size=20,
        dropout=0.1,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2026/03/06 14:38:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/06 14:38:56 INFO mlflow.store.db.utils: Updating database tables
2026-03-06 14:38:56 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-03-06 14:38:56 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026-03-06 14:38:56 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2026-03-06 14:38:56 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
c:\Users\Alex\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1334: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("GRU", *args, **kwargs)
2026-03-06 14:38:56.996 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260306-143856
2026-03-06 14:38:59.248 | INFO     | mltrainer.trainer:__init__:66 - Fou

Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

In [ ]:
trainer.loop() # if you want to pick up training, loop will continue from the last epoch

In [ ]:
mlflow.end_run()

## Alextuning

In [ ]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat





In [ ]:
settings = TrainerSettings(
    epochs=100,
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    optimizer_kwargs={"lr": lr},   # ← hier
    earlystop_kwargs={
        "save": False,
        "verbose": True,
        "patience": 5,
        "delta": 0.0,
    }
)
settings

In [ ]:
import random
import mlflow
from datetime import datetime
from pathlib import Path
import torch
import torch.optim as optim

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")

modeldir = Path("gestures").resolve()
modeldir.mkdir(parents=True, exist_ok=True)

hidden_sizes = [32, 64, 128]
num_layers_options = [1, 2, 3]
dropouts = [0.0, 0.1, 0.2, 0.3]
learning_rates = [0.001, 0.0005, 0.0001]

n_trials = 30

for trial in range(n_trials):

    hidden_size = random.choice(hidden_sizes)
    num_layers = random.choice(num_layers_options)
    dropout = random.choice(dropouts)
    lr = random.choice(learning_rates)

    with mlflow.start_run():

        mlflow.set_tag("model", "gru-gesture")
        mlflow.set_tag("trial", trial)

        mlflow.log_params({
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "learning_rate": lr
        })

        config = ModelConfig(
            input_size=3,
            hidden_size=hidden_size,
            num_layers=num_layers,
            output_size=20,
            dropout=dropout,
        )

        model = GRUmodel(config=config)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )

        trainer.loop()

        trial_name = f"trial{trial}_h{hidden_size}_l{num_layers}_d{int(dropout*100)}_lr{lr:.0e}.pt"
        modelpath = modeldir / trial_name
        torch.save(model, modelpath)

## CONV1D

In [25]:
import torch.nn as nn
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class Conv1d_RNN(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=config.input_size,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.rnn = nn.GRU(
            input_size=32,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout,
            batch_first=True
        )

        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor):

        x = x.transpose(1, 2)

        x = self.conv1(x)
        x = self.relu(x)

        x = x.transpose(1, 2)

        x, _ = self.rnn(x)

        last_step = x[:, -1, :]

        yhat = self.linear(last_step)

        return yhat

In [30]:
from pathlib import Path
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

lr = 0.001

accuracy = Accuracy()

settings = TrainerSettings(
    epochs=100,
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    optimizer_kwargs={"lr": lr},
    earlystop_kwargs={
        "save": False,
        "verbose": True,
        "patience": 5,
        "delta": 0.0,
    }
)

In [35]:
import mlflow
from pathlib import Path
import torch
import torch.optim as optim
from functools import partial

optimizer_fn = partial(optim.Adam, lr=cfg["lr"])

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")

modeldir = Path("gestures").resolve()
modeldir.mkdir(parents=True, exist_ok=True)

model_configs = [
    {"name": "Intrigued fly 96", "hidden_size":128, "num_layers":2, "dropout":0.2, "lr":0.001},
    {"name": "Painted Moth 165", "hidden_size":128, "num_layers":3, "dropout":0.3, "lr":0.0001},
    {"name": "Blushing newt 123", "hidden_size":128, "num_layers":3, "dropout":0.1, "lr":0.0001},
    {"name": "Victorious Hound 539", "hidden_size":128, "num_layers":3, "dropout":0.0, "lr":0.001},
    {"name": "Unleashed Gnu 130", "hidden_size":128, "num_layers":3, "dropout":0.1, "lr":0.0001}
]

for trial, cfg in enumerate(model_configs):

    with mlflow.start_run():
        mlflow.set_tag("model", "conv1d-gru-gesture")
        mlflow.set_tag("trial", trial)
        mlflow.set_tag("name", cfg["name"])
        mlflow.log_params(cfg)

        config = ModelConfig(
            input_size=3,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            output_size=20,
            dropout=cfg["dropout"],
        )
        model = Conv1d_RNN(config=config)

        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optimizer_fn,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )

        trainer.loop()

        trial_name = (
            f"{cfg['name'].replace(' ','_')}_"
            f"h{cfg['hidden_size']}_"
            f"l{cfg['num_layers']}_"
            f"d{int(cfg['dropout']*100)}_"
            f"lr{cfg['lr']:.0e}.pt"
        )
        modelpath = modeldir / trial_name
        torch.save(model, modelpath)
        mlflow.log_artifact(modelpath)

 

2026-03-06 14:46:09.810 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260306-144609
2026-03-06 14:46:09.811 | INFO     | mltrainer.trainer:__init__:66 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:02<00:00, 31.42it/s]
2026-03-06 14:46:12.574 | INFO     | mltrainer.trainer:report:215 - Epoch 0 train 2.6274 test 2.2234 metric ['0.1875']
100%|██████████| 81/81 [00:02<00:00, 31.39it/s]
2026-03-06 14:46:15.333 | INFO     | mltrainer.trainer:report:215 - Epoch 1 train 1.9805 test 1.5774 metric ['0.3781']
100%|██████████| 81/81 [00:02<00:00, 32.91it/s]
2026-03-06 14:46:17.970 | INFO     | mltrainer.trainer:report:215 - Epoch 2 train 1.2916 test 0.8658 metric ['0.7016']
100%|██████████| 81/81 [00:02<00:00, 33.27it/s]
2026-03-06 14:46:20.575 | INFO     | mltrainer.trainer:report:215 - Epoch 3 train 0.6801 test 0.4740 metric ['0.8781']
100%|██████████| 81/81 [00:02<00:00, 34.00it/s]
2026-03-06 14:46:23.12